In [1]:
from itertools import zip_longest

import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

from mpnn import mol_to_torch, visualize_activations, visualize_all_activations
from matplotlib import pyplot as plt

from krfp_models import krfp_models, model_name_to_publication_name


/home/dominik/MyStuff/bxaic2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pathlib

base_path = pathlib.Path("notebook_artifacts/activations_visualisation")

for model, smarts, _ in krfp_models:
    img_dir = base_path / model_name_to_publication_name[model.__class__.__name__]
    img_dir.mkdir(parents=True, exist_ok=True)

    mol = Chem.MolFromSmarts(smarts)
    smiles = Chem.MolToSmiles(mol, canonical=True)
    smiles = smiles.replace("*", "C")

    mol = Chem.MolFromSmiles(smiles)

    x, edge_index, edge_features = mol_to_torch(mol, add_hydrogen_ohe=True)
    _, activations = model(x, edge_features, edge_index, dump_activations=True)

    imgs = visualize_all_activations(mol, activations, bond_line_width=8, annotation_font_scale=1.1, length=1000)

    for i, img in enumerate(imgs):
        img.save(img_dir / f"layer_{i + 1}.png")

In [5]:
from mpnn import MoleculeODetector

mol = Chem.MolFromSmiles("[CH-]=O")

x, edge_index, edge_features = mol_to_torch(mol, add_hydrogen_ohe=True)
_, activations = MoleculeODetector()(x, edge_features, edge_index, dump_activations=True)

imgs = visualize_all_activations(mol, activations, bond_line_width=8, annotation_font_scale=1.1, length=1000)

path = pathlib.Path("notebook_artifacts/Patter12Carbonyl/activations.png")
path.parent.mkdir(parents=True, exist_ok=True)
imgs[0].save(path)